# Análisis de  Aerogeneradores - SCADA PEM
Este notebook contiene gráficas relevantes generadas a partir de `1.SCADA_PEM_horario.csv`.

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Cargando datos...")
df = pd.read_csv('1.SCADA_PEM_con_PT.csv', sep=';')
df['FechaHora'] = pd.to_datetime(df['FechaHora'])

# Crear columna de Mes para análisis mensual (ej. '2026-01', '2026-02')
df['Mes'] = df['FechaHora'].dt.to_period('M').astype(str)

# Corregir unidades: ActivePower viene en kW y Potencia Disponible en MW.
# Convertimos ActivePower a MW dividiendo entre 1000.
df['ActivePower_MW'] = df['ActivePower'] / 1000

df.head()

Cargando datos...


,FechaHora,Name,StationTypeId,WindSpeed,ActivePower,ActivePower_MW,OperationCode,Potencia Teórica,Potencia Perdida,Error (%),Mes
0,2026-01-01,M-01,5.0,9.504695,2051.437744,2.051438,Normal,2.032535,0.000000,0.929995,2026-01
1,2026-01-01,M-02,5.0,9.483687,1899.135620,1.899136,Normal,2.021191,0.122056,6.038794,2026-01
2,2026-01-01,M-03,5.0,8.777123,1602.170044,1.602170,Normal,1.646333,0.044163,2.682502,2026-01
3,2026-01-01,M-04,4.0,8.219446,1489.326782,1.489327,Normal,1.339723,0.000000,11.166762,2026-01
4,2026-01-01,M-05,4.0,6.791320,228.422531,0.228423,Normal,0.745309,0.516887,69.351976,2026-01


## Gráfica Interactiva: Serie Temporal por Aerogenerador
Esta gráfica interactiva (usando Plotly) permite seleccionar un aerogenerador específico y visualizar su `ActivePower_MW`, `Potencia Teórica` y el `Error (%)` a lo largo del tiempo.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

aerogeneradores = sorted(df['Name'].dropna().unique())
meses = sorted(df['Mes'].dropna().unique())

n_aeros = len(aerogeneradores)
n_meses = len(meses)

# 2 filas × 3 columnas (una columna por mes)
fig = make_subplots(
    rows=2, cols=n_meses,
    shared_xaxes=False,
    vertical_spacing=0.12,
    horizontal_spacing=0.06,
    row_heights=[0.65, 0.35],
    subplot_titles=(
        [f"Potencia (MW) — {mes}" for mes in meses] +
        [f"Error (%) — {mes}" for mes in meses]
    )
)

# Añadir trazas: para cada aerogenerador × mes
for a_idx, aero in enumerate(aerogeneradores):
    is_first = (a_idx == 0)
    for m_idx, mes in enumerate(meses):
        col = m_idx + 1
        df_ag = df[(df['Mes'] == mes) & (df['Name'] == aero)].sort_values('FechaHora')
        show_leg = (is_first and m_idx == 0)

        fig.add_trace(go.Scatter(
            x=df_ag['FechaHora'], y=df_ag['ActivePower_MW'],
            mode='lines', name='Potencia Real (MW)',
            visible=is_first,
            line=dict(color='royalblue'),
            showlegend=show_leg,
            legendgroup='real'
        ), row=1, col=col)

        fig.add_trace(go.Scatter(
            x=df_ag['FechaHora'], y=df_ag['Potencia Teórica'],
            mode='lines', name='Potencia Teórica (MW)',
            visible=is_first,
            line=dict(color='dimgray', dash='dash'),
            showlegend=show_leg,
            legendgroup='teorica'
        ), row=1, col=col)

        fig.add_trace(go.Scatter(
            x=df_ag['FechaHora'], y=df_ag['Error (%)'],
            mode='lines', name='Error (%)',
            visible=is_first,
            line=dict(color='crimson'),
            showlegend=show_leg,
            legendgroup='error'
        ), row=2, col=col)

# Total trazas: n_aeros × n_meses × 3
total_traces = n_aeros * n_meses * 3

# Botones del dropdown por aerogenerador
aero_buttons = []
for a_idx, aero in enumerate(aerogeneradores):
    visibility = [False] * total_traces
    base = a_idx * n_meses * 3
    for i in range(n_meses * 3):
        visibility[base + i] = True

    aero_buttons.append(dict(
        label=aero,
        method='update',
        args=[
            {'visible': visibility},
            {'title': {
                'text': 'Análisis de Curvas de Potencia Real vs Teórico',
                'x': 0.5,
                'xanchor': 'center'
            }}
        ]
    ))

fig.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=aero_buttons,
        x=1.0,
        y=1.12,
        xanchor='right',
        yanchor='top',
        type='dropdown',
        bgcolor='white',
        bordercolor='gray'
    )],
    title=dict(
        text='Análisis de Curvas de Potencia Real vs Teórico',
        x=0.5,
        xanchor='center',
        font=dict(size=16)
    ),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.18,
        xanchor="center",
        x=0.5
    ),
    template='plotly_white',
    height=780,
    width=1400,
    margin=dict(b=110)
)

# Configurar ejes X para cada columna
for col in range(1, n_meses + 1):
    fig.update_xaxes(
        dtick=86400000.0,
        tickformat="%d-%b",
        tickangle=-45,
        title_text="Días del Mes",
        row=2, col=col
    )
    fig.update_xaxes(
        dtick=86400000.0,
        tickformat="%d-%b",
        row=1, col=col
    )

fig.add_annotation(
    x=0.915, y=1.09,
    xref='paper', yref='paper',
    xanchor='right',
    showarrow=False,
    text='Aerogenerador:'
)

fig.show()
